[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/30_cosine_lr_solution.ipynb)

# ✅ Solution: Cosine LR Scheduler with Warmup

linear warmup 付きの **cosine learning rate schedule** を実装する。


> 💡 **どこで使う**：モダンな学習の LR スケジューラの標準。warmup で発散を防ぎ、cosine で滑らかに 0 へ収束。

<details>
<summary>📖 もっと詳しく</summary>

ResNet, BERT, GPT 系の論文で見る形。warmup steps は全 step の 5–10%、min_lr=0 が多い。

注意: step ベース (epoch ベースじゃない)、step ≠ batch — gradient accumulation (#31) する場合は accumulation 後の optimizer step だけカウント。

</details>

### Signature
```python
def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0) -> float:
```

<details>
<summary>📐 アルゴリズム (Schedule)</summary>

```
step < warmup:  lr = max_lr * step / warmup_steps  (linear ramp)
step >= warmup: lr = min_lr + 0.5*(max_lr-min_lr)*(1 + cos(π * progress))
```
ここで `progress = (step - warmup) / (total - warmup)`

</details>


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass


In [ ]:
import math

In [ ]:
# ✅ SOLUTION

def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step >= total_steps:
        return min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

In [ ]:
# Demo
lrs = [cosine_lr_schedule(i, 100, 10, 0.001) for i in range(101)]
print(f'Start: {lrs[0]:.6f}, Warmup end: {lrs[10]:.6f}, Mid: {lrs[55]:.6f}, End: {lrs[100]:.6f}')

In [ ]:
from torch_judge import check
check('cosine_lr')